<a href="https://colab.research.google.com/github/lucas-estanislau/clima-e-escorpioes/blob/coleta/01_coleta_clima.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [14]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"

parametros = {
    "latitude": -5.7945,
    "longitude": -35.211,
    "start_date": "2007-01-01",
    "end_date": "2007-12-31",
    "daily": "temperature_2m_mean",
    "timezone": "auto"
}

resposta = requests.get(url, params=parametros)
resposta.raise_for_status()

dados = resposta.json()

df = pd.DataFrame(dados["daily"])

display(df.head())

,time,temperature_2m_mean
0,2007-01-01,26.1
1,2007-01-02,26.7
2,2007-01-03,26.3
3,2007-01-04,25.7
4,2007-01-05,26.2


In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import os
import json
import requests
import pandas as pd

pasta = "/content/drive/MyDrive/DataLake"

url = "https://archive-api.open-meteo.com/v1/archive"
parametros = {
    "latitude": -5.7945,
    "longitude": -35.211,
    "start_date": "2007-01-01",
    "end_date": "2007-12-31",
    "daily": "temperature_2m_mean",
    "timezone": "auto"
}

resposta = requests.get(url, params=parametros)
resposta.raise_for_status()
dados = resposta.json()
df = pd.DataFrame(dados["daily"])

In [17]:
os.makedirs(pasta + "/brutos/clima", exist_ok=True)
os.makedirs(pasta + "/processados/clima", exist_ok=True)

with open(pasta + "/brutos/clima/natal_2007.json", "w") as arquivo:
    json.dump(dados, arquivo)

df.to_csv(pasta + "/processados/clima/natal_2007.csv", index=False)

print("Arquivos climáticos salvos!")

Arquivos climáticos salvos!


---
#Automatizando

In [20]:
import pandas as pd

url = "https://raw.githubusercontent.com/lizmckenna/Municipios-Brasileiros/master/municipios_brasileiros.csv"

municipios = pd.read_csv(url)
rn = municipios[municipios["uf"] == "RN"].copy()

display(rn.head())
print("Total de municípios:", len(rn))


,codigo_ibge,nome_municipio,codigo_uf,uf,estado,capital,latitude,longitude
21,2400109,Acari,24,RN,Rio Grande do Norte,0,-6.42820,-36.6347
28,2400208,Açu,24,RN,Rio Grande do Norte,0,-5.58362,-36.9140
36,2400307,Afonso Bezerra,24,RN,Rio Grande do Norte,0,-5.49229,-36.5075
59,2400406,Água Nova,24,RN,Rio Grande do Norte,0,-6.20351,-38.2941
108,2400505,Alexandria,24,RN,Rio Grande do Norte,0,-6.40533,-38.0142


Total de municípios: 167


In [21]:
cidades = ["Natal", "Mossoró", "Caicó"]

filtro = rn.astype(str).isin(cidades).any(axis=1)
display(rn[filtro])


,codigo_ibge,nome_municipio,codigo_uf,uf,estado,capital,latitude,longitude
887,2402006,Caicó,24,RN,Rio Grande do Norte,0,-6.45441,-37.1067
3197,2408003,Mossoró,24,RN,Rio Grande do Norte,0,-5.18374,-37.3474
3239,2408102,Natal,24,RN,Rio Grande do Norte,1,-5.79357,-35.1986


In [23]:
import requests
import pandas as pd
import time

url = "https://archive-api.open-meteo.com/v1/archive"

tabelas = []
falhas = []

for _, municipio in rn.iterrows():

    nome = municipio["nome_municipio"]

    parametros = {
        "latitude": municipio["latitude"],
        "longitude": municipio["longitude"],
        "start_date": "2007-01-01",
        "end_date": "2025-12-31",
        "daily": "temperature_2m_mean",
        "timezone": "auto",
        "models": "era5_land"
    }

    try:
        resposta = requests.get(url, params=parametros, timeout=120)

        if resposta.status_code == 429:
            print("Limite da API atingido. Coleta interrompida.")
            break

        resposta.raise_for_status()

        dados = resposta.json()
        df = pd.DataFrame(dados["daily"])

        df["municipio"] = nome
        df["codigo_ibge"] = municipio["codigo_ibge"]

        tabelas.append(df)

        print("Coletado:", nome)

    except requests.RequestException as erro:
        falhas.append(nome)
        print("Erro em:", nome, erro)

    time.sleep(15)

if tabelas:
    df_clima_rn = pd.concat(tabelas, ignore_index=True)

    print("Municípios coletados:", len(tabelas))
    print("Municípios com falha:", falhas)
    print("Total de registros:", len(df_clima_rn))
else:
    print("Nenhum município foi coletado.")

Coletado: Acari
Coletado: Açu
Coletado: Afonso Bezerra
Coletado: Água Nova
Coletado: Alexandria
Coletado: Almino Afonso
Coletado: Alto do Rodrigues
Coletado: Angicos
Coletado: Antônio Martins
Coletado: Apodi
Coletado: Areia Branca
Coletado: Arês
Coletado: Augusto Severo (Campo Grande)
Coletado: Baía Formosa
Coletado: Baraúna
Coletado: Barcelona
Coletado: Bento Fernandes
Coletado: Bodó
Coletado: Bom Jesus
Coletado: Brejinho
Coletado: Caiçara do Norte
Coletado: Caiçara do Rio do Vento
Coletado: Caicó
Coletado: Campo Redondo
Coletado: Canguaretama
Coletado: Caraúbas
Coletado: Carnaúba dos Dantas
Coletado: Carnaubais
Coletado: Ceará-Mirim
Coletado: Cerro Corá
Coletado: Coronel Ezequiel
Coletado: Coronel João Pessoa
Coletado: Cruzeta
Coletado: Currais Novos
Coletado: Doutor Severiano
Coletado: Encanto
Coletado: Equador
Coletado: Espírito Santo
Coletado: Extremoz
Coletado: Felipe Guerra
Coletado: Fernando Pedroza
Coletado: Florânia
Coletado: Francisco Dantas
Coletado: Frutuoso Gomes
Coletado

In [24]:
pasta = "/content/drive/MyDrive/DataLake/processados/clima"
os.makedirs(pasta, exist_ok=True)

arquivo = os.path.join(pasta, "rn_2007_2025_parcial.csv")

df_clima_rn.to_csv(arquivo, index=False)

print("Arquivo parcial salvo!")
print("Municípios:", df_clima_rn["codigo_ibge"].nunique())
print("Registros:", len(df_clima_rn))

Arquivo parcial salvo!
Municípios: 101
Registros: 700940


In [25]:
codigos_coletados = set(
    df_clima_rn["codigo_ibge"].astype(str)
)

rn_restantes = rn[
    ~rn["codigo_ibge"].astype(str).isin(codigos_coletados)
].copy()

print("Municípios restantes:", len(rn_restantes))
display(rn_restantes.head())

Municípios restantes: 66


,codigo_ibge,nome_municipio,codigo_uf,uf,estado,capital,latitude,longitude
3696,2409506,Pedra Grande,24,RN,Rio Grande do Norte,0,-5.14988,-35.8760
3699,2409605,Pedra Preta,24,RN,Rio Grande do Norte,0,-5.57352,-36.1084
3716,2409704,Pedro Avelino,24,RN,Rio Grande do Norte,0,-5.51610,-36.3867
3727,2409803,Pedro Velho,24,RN,Rio Grande do Norte,0,-6.43560,-35.2195
3736,2409902,Pendências,24,RN,Rio Grande do Norte,0,-5.25640,-36.7095


In [26]:
pasta = "/content/drive/MyDrive/DataLake/processados/clima"
arquivo = f"{pasta}/rn_2007_2025_parcial.csv"

# Recuperar os dados já salvos no Drive
df_clima_rn = pd.read_csv(arquivo, dtype={"codigo_ibge": str})

# Identificar quem ainda falta
coletados = set(df_clima_rn["codigo_ibge"])

restantes = rn[
    ~rn["codigo_ibge"].astype(str).isin(coletados)
]

print("Municípios restantes:", len(restantes))

url = "https://archive-api.open-meteo.com/v1/archive"

for _, municipio in restantes.iterrows():
    nome = municipio["nome_municipio"]

    parametros = {
        "latitude": municipio["latitude"],
        "longitude": municipio["longitude"],
        "start_date": "2007-01-01",
        "end_date": "2025-12-31",
        "daily": "temperature_2m_mean",
        "timezone": "auto",
        "models": "era5_land"
    }

    try:
        resposta = requests.get(
            url, params=parametros, timeout=120
        )

        if resposta.status_code == 429:
            print("Limite da API atingido. Parando.")
            break

        resposta.raise_for_status()

        dados = resposta.json()
        df = pd.DataFrame(dados["daily"])

        df["municipio"] = nome
        df["codigo_ibge"] = str(municipio["codigo_ibge"])

        # Adicionar ao resultado acumulado
        df_clima_rn = pd.concat(
            [df_clima_rn, df],
            ignore_index=True
        )

        # Salvar progresso após cada município
        df_clima_rn.to_csv(arquivo, index=False)

        print("Coletado e salvo:", nome)

    except requests.RequestException as erro:
        print("Erro em:", nome, erro)
        break

    time.sleep(15)

print(
    "Total de municípios:",
    df_clima_rn["codigo_ibge"].nunique()
)

Municípios restantes: 66
Limite da API atingido. Parando.
Total de municípios: 101
